# Step 9: Spark Internal Architecture

## Learning Objectives
1. Spark application execution structure (Driver, Executor, Cluster Manager)
2. Job → Stage → Task splitting principles
3. DAG Scheduler: Stage boundaries and Shuffle dependencies
4. Task Scheduler: data locality and scheduling
5. BlockManager and data flow
6. Observing internal behavior in the Spark UI
7. Lineage and fault recovery

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
from datetime import datetime, timedelta
import random
import time

spark = SparkSession.builder \
    .appName('Step9-Spark-Internals') \
    .master('spark://spark-master:7077') \
    .config('spark.executor.memory', '1g') \
    .config('spark.executor.cores', '1') \
    .config('spark.sql.shuffle.partitions', '10') \
    .config('spark.sql.adaptive.enabled', 'false') \
    .config('spark.sql.warehouse.dir', '/home/jovyan/data/warehouse') \
    .getOrCreate()

sc = spark.sparkContext
print(f'App ID:     {sc.applicationId}')
print(f'Master:     {sc.master}')
print(f'Executors:  {sc.defaultParallelism}')
print(f'✅ Spark UI: http://localhost:4040')

App ID:     app-20260603094515-0032
Master:     spark://spark-master:7077
Executors:  2
✅ Spark UI: http://localhost:4040


---
## 1. Spark Application Execution Structure

```
┌─────────────────────────────────────────────────────────────────┐
│                     Cluster Manager                            │
│                  (Standalone / YARN / K8s)                     │
│                                                                │
│  ┌──────────────────────┐    ┌───────────────────────────┐     │
│  │      Driver          │    │     Executor 1            │     │
│  │                      │    │  ┌──────┐ ┌──────┐       │     │
│  │  SparkContext         │    │  │Task 1│ │Task 2│       │     │
│  │  DAG Scheduler       │    │  └──────┘ └──────┘       │     │
│  │  Task Scheduler      │───→│  Block Manager            │     │
│  │  UI (4040)           │    │  Shuffle Manager          │     │
│  │                      │    └───────────────────────────┘     │
│  │                      │                                      │
│  │                      │    ┌───────────────────────────┐     │
│  │                      │    │     Executor 2            │     │
│  │                      │    │  ┌──────┐ ┌──────┐       │     │
│  │                      │───→│  │Task 3│ │Task 4│       │     │
│  │                      │    │  └──────┘ └──────┘       │     │
│  └──────────────────────┘    │  Block Manager            │     │
│                              └───────────────────────────┘     │
└─────────────────────────────────────────────────────────────────┘
```

### Component Roles
| Component | Role |
|-----------|------|
| **Driver** | Runs user code, creates execution plan, schedules Tasks |
| **Executor** | Runs Tasks, stores data (cache), manages Shuffle data |
| **Cluster Manager** | Allocates resources (Standalone, YARN, K8s, Mesos) |
| **Task** | Execution unit that processes one partition |

---
## 2. Job → Stage → Task Splitting Principles

```
Action call (collect, count, save...)
    │
    ▼
┌─────────┐
│   Job   │  1 Action = 1 Job
└────┬────┘
     │  split at Shuffle boundaries
     ▼
┌─────────┐  ┌─────────┐  ┌─────────┐
│ Stage 0 │→│ Stage 1 │→│ Stage 2 │
│ (read)  │  │(shuffle)│  │ (agg)   │
└────┬────┘  └────┬────┘  └────┬────┘
     │            │            │
     ▼            ▼            ▼
  Task 0-0     Task 1-0     Task 2-0
  Task 0-1     Task 1-1     Task 2-1
  Task 0-2     Task 1-2     Task 2-2
  ...          ...          ...
  (one Task per partition)
```

**Key rules:**
- 1 **Action** = 1 **Job**
- **Shuffle (Wide Dependency)** boundary = **Stage** boundary
- 1 **Partition** = 1 **Task**

In [2]:
# Experiment: observe Job, Stage, and Task counts
random.seed(42)
data = [(i, random.choice(['A','B','C','D','E']), random.randint(1, 1000)) 
        for i in range(100_000)]
df = spark.createDataFrame(data, ['id', 'group', 'value'])

print('=== Experiment 1: Simple filter + count (no Shuffle) ===')
result1 = df.filter(F.col('value') > 500).count()
print(f'Result: {result1:,}')
print('→ Check in Spark UI: 1 Job, 1 Stage')
print(f'  (partitions = tasks = {df.rdd.getNumPartitions()})')

print()
print('=== Experiment 2: groupBy + count (1 Shuffle) ===')
result2 = df.groupBy('group').count().collect()
print(f'Result: {result2}')
print('→ Check in Spark UI: 1 Job, 2 Stages')
print('  Stage 0: map phase (local aggregation per partition)')
print('  Stage 1: reduce phase (final aggregation after Shuffle)')

print()
print('=== Experiment 3: groupBy + orderBy (2 Shuffles) ===')
result3 = df.groupBy('group').agg(F.avg('value').alias('avg_val')) \
    .orderBy(F.col('avg_val').desc()).collect()
print(f'Result: {result3}')
print('→ Check in Spark UI: 1 Job, 3 Stages')
print('  Stage 0: map (read source + local aggregation)')
print('  Stage 1: Shuffle 1 (merge groupBy results)')
print('  Stage 2: Shuffle 2 (sort for orderBy)')

=== Experiment 1: Simple filter + count (no Shuffle) ===
Result: 49,955
→ Check in Spark UI: 1 Job, 1 Stage
  (partitions = tasks = 2)

=== Experiment 2: groupBy + count (1 Shuffle) ===
Result: [Row(group='E', count=19968), Row(group='A', count=19862), Row(group='D', count=20247), Row(group='C', count=19915), Row(group='B', count=20008)]
→ Check in Spark UI: 1 Job, 2 Stages
  Stage 0: map phase (local aggregation per partition)
  Stage 1: reduce phase (final aggregation after Shuffle)

=== Experiment 3: groupBy + orderBy (2 Shuffles) ===
Result: [Row(group='C', avg_val=502.06547828270146), Row(group='E', avg_val=501.48823116987177), Row(group='D', avg_val=500.7035610213859), Row(group='A', avg_val=498.5667606484745), Row(group='B', avg_val=496.3815973610556)]
→ Check in Spark UI: 1 Job, 3 Stages
  Stage 0: map (read source + local aggregation)
  Stage 1: Shuffle 1 (merge groupBy results)
  Stage 2: Shuffle 2 (sort for orderBy)


In [3]:
# Directly inspect RDD Lineage (dependency graph)
rdd = sc.parallelize(range(100), 4) \
    .map(lambda x: (x % 5, x)) \
    .groupByKey() \
    .mapValues(list)

print('=== RDD Lineage (toDebugString) ===')
print(rdd.toDebugString().decode('utf-8'))

print('''
How to read:
  (N) = number of partitions
  indentation = dependency direction (child → parent)
  ShuffledRDD = where Shuffle occurs = Stage boundary
''')

=== RDD Lineage (toDebugString) ===
(4) PythonRDD[35] at RDD at PythonRDD.scala:53 []
 |  MapPartitionsRDD[34] at mapPartitions at PythonRDD.scala:160 []
 |  ShuffledRDD[33] at partitionBy at NativeMethodAccessorImpl.java:0 []
 +-(4) PairwiseRDD[32] at groupByKey at /tmp/ipykernel_37786/2760595311.py:4 []
    |  PythonRDD[31] at groupByKey at /tmp/ipykernel_37786/2760595311.py:4 []
    |  ParallelCollectionRDD[30] at readRDDFromFile at PythonRDD.scala:289 []

How to read:
  (N) = number of partitions
  indentation = dependency direction (child → parent)
  ShuffledRDD = where Shuffle occurs = Stage boundary



---
## 3. DAG Scheduler Deep Dive

The DAG Scheduler **converts the logical execution plan (RDD lineage) into a physical execution plan (Stages)**.

### Narrow vs Wide Dependency
```
Narrow (1:1 or N:1)            Wide (N:N, Shuffle)
  Parent    Child              Parent    Child
  ┌───┐    ┌───┐              ┌───┐    ┌───┐
  │ P0│───→│ C0│              │ P0│╲  ╱│ C0│
  └───┘    └───┘              └───┘ ╲╱ └───┘
  ┌───┐    ┌───┐              ┌───┐ ╱╲ ┌───┐
  │ P1│───→│ C1│              │ P1│╱  ╲│ C1│
  └───┘    └───┘              └───┘    └───┘
  
  pipelining possible            Stage boundary!
  within the same Stage          data redistribution required
```

In [4]:
# Narrow vs Wide dependency experiment

# Narrow chain: map → filter → map (1 Stage)
narrow_rdd = sc.parallelize(range(10000), 4) \
    .map(lambda x: x * 2) \
    .filter(lambda x: x % 3 == 0) \
    .map(lambda x: (x, x ** 0.5))

print('=== Narrow chain ===')
print(narrow_rdd.toDebugString().decode('utf-8'))
print(f'→ No ShuffledRDD = 1 Stage')
print(f'→ All map/filter ops are pipelined (fused)')

print()

# Wide dependency: map → groupByKey → mapValues (2 Stages)
wide_rdd = sc.parallelize(range(10000), 4) \
    .map(lambda x: (x % 10, x)) \
    .groupByKey() \
    .mapValues(lambda vals: sum(vals))

print('=== Wide dependency ===')
print(wide_rdd.toDebugString().decode('utf-8'))
print(f'→ ShuffledRDD present = 2 Stages')

=== Narrow chain ===
(4) PythonRDD[37] at RDD at PythonRDD.scala:53 []
 |  ParallelCollectionRDD[36] at readRDDFromFile at PythonRDD.scala:289 []
→ No ShuffledRDD = 1 Stage
→ All map/filter ops are pipelined (fused)

=== Wide dependency ===
(4) PythonRDD[43] at RDD at PythonRDD.scala:53 []
 |  MapPartitionsRDD[42] at mapPartitions at PythonRDD.scala:160 []
 |  ShuffledRDD[41] at partitionBy at NativeMethodAccessorImpl.java:0 []
 +-(4) PairwiseRDD[40] at groupByKey at /tmp/ipykernel_37786/3645943007.py:19 []
    |  PythonRDD[39] at groupByKey at /tmp/ipykernel_37786/3645943007.py:19 []
    |  ParallelCollectionRDD[38] at readRDDFromFile at PythonRDD.scala:289 []
→ ShuffledRDD present = 2 Stages


In [5]:
# Observe Stage splitting at the DataFrame level
# Exchange node in explain() = Stage boundary

df1 = spark.range(100000).withColumn('group', (F.col('id') % 10).cast('string'))
df2 = spark.range(100000).withColumn('group', (F.col('id') % 10).cast('string'))

# Complex query: join + groupBy + orderBy
complex_query = df1.join(df2.hint('MERGE'), 'group') \
    .groupBy(df1.group) \
    .agg(F.count('*').alias('cnt')) \
    .orderBy('cnt')

print('=== Execution plan for complex query ===')
complex_query.explain()

print('''
Number of Exchange nodes = number of Shuffles = number of Stage boundaries

Stage breakdown for this query:
  Stage 0: scan df1
  Stage 1: scan df2
  Stage 2: Exchange (Shuffle for Join) → SortMergeJoin
  Stage 3: Exchange (Shuffle for groupBy) → HashAggregate
  Stage 4: Exchange (Shuffle for orderBy) → Sort
''')

=== Execution plan for complex query ===
== Physical Plan ==
*(6) Sort [cnt#57L ASC NULLS FIRST], true, 0
+- Exchange rangepartitioning(cnt#57L ASC NULLS FIRST, 10), ENSURE_REQUIREMENTS, [plan_id=213]
   +- *(5) HashAggregate(keys=[group#42], functions=[count(1)])
      +- *(5) HashAggregate(keys=[group#42], functions=[partial_count(1)])
         +- *(5) Project [group#42]
            +- *(5) SortMergeJoin [group#42], [group#47], Inner
               :- *(2) Sort [group#42 ASC NULLS FIRST], false, 0
               :  +- Exchange hashpartitioning(group#42, 10), ENSURE_REQUIREMENTS, [plan_id=167]
               :     +- *(1) Project [cast((id#40L % 10) as string) AS group#42]
               :        +- *(1) Filter isnotnull(cast((id#40L % 10) as string))
               :           +- *(1) Range (0, 100000, step=1, splits=4)
               +- *(4) Sort [group#47 ASC NULLS FIRST], false, 0
                  +- ReusedExchange [group#47], Exchange hashpartitioning(group#42, 10), ENSURE_REQUI

---
## 4. Task Scheduler & Data Locality

The Task Scheduler tries to run each Task **on the node closest to the data**.

```
Locality priority (highest first):

  PROCESS_LOCAL  → data is in the same JVM (cached)
       ↓
  NODE_LOCAL     → data is on the same node's disk
       ↓
  RACK_LOCAL     → data is on a different node in the same rack
       ↓
  ANY            → anywhere (network transfer required)
```

Related settings:
- `spark.locality.wait` = 3s (default): time to wait for higher locality
- `spark.locality.wait.process`, `.node`, `.rack`: per-level wait times

In [6]:
# Check locality-related settings
locality_configs = [
    'spark.locality.wait',
    'spark.locality.wait.process',
    'spark.locality.wait.node',
    'spark.locality.wait.rack',
]

print('=== Data Locality Settings ===')
for key in locality_configs:
    try:
        val = spark.conf.get(key)
    except Exception:
        val = '(default: 3s)'
    print(f'  {key} = {val}')

print('''
💡 Locality and performance:
   PROCESS_LOCAL: memory access only → fastest
   NODE_LOCAL:    local disk I/O → fast
   ANY:           network transfer → slow

   In Spark UI > Stages > Task details,
   you can check the Locality Level for each Task.
''')

# Demo: cache → PROCESS_LOCAL
test_df = spark.range(500_000).withColumn('v', F.rand())

# Without cache
start = time.time()
test_df.groupBy((F.col('id') % 5).alias('g')).agg(F.sum('v')).collect()
no_cache = time.time() - start

# With cache → PROCESS_LOCAL
test_df.cache().count()
start = time.time()
test_df.groupBy((F.col('id') % 5).alias('g')).agg(F.sum('v')).collect()
with_cache = time.time() - start

test_df.unpersist()

print(f'Without cache: {no_cache:.3f}s (ANY/NODE_LOCAL)')
print(f'With cache:    {with_cache:.3f}s (PROCESS_LOCAL)')

=== Data Locality Settings ===
  spark.locality.wait = (default: 3s)
  spark.locality.wait.process = (default: 3s)
  spark.locality.wait.node = (default: 3s)
  spark.locality.wait.rack = (default: 3s)

💡 Locality and performance:
   PROCESS_LOCAL: memory access only → fastest
   NODE_LOCAL:    local disk I/O → fast
   ANY:           network transfer → slow

   In Spark UI > Stages > Task details,
   you can check the Locality Level for each Task.

Without cache: 0.361s (ANY/NODE_LOCAL)
With cache:    0.247s (PROCESS_LOCAL)


---
## 5. BlockManager and Data Flow

```
┌────────────────────────────────────────────┐
│              Executor                       │
│                                            │
│  ┌──────────────────────────────────────┐  │
│  │          Block Manager               │  │
│  │                                      │  │
│  │  ┌─────────────┐  ┌──────────────┐  │  │
│  │  │ Memory Store│  │  Disk Store  │  │  │
│  │  │             │  │              │  │  │
│  │  │ RDD cache   │  │ Spill data   │  │  │
│  │  │ Broadcast   │  │ Shuffle files│  │  │
│  │  │ Task results│  │              │  │  │
│  │  └─────────────┘  └──────────────┘  │  │
│  │                                      │  │
│  │  ┌──────────────────────────────┐   │  │
│  │  │     Shuffle Manager          │   │  │
│  │  │  Shuffle Write → local files │   │  │
│  │  │  Shuffle Read  ← remote fetch│   │  │
│  │  └──────────────────────────────┘   │  │
│  └──────────────────────────────────────┘  │
└────────────────────────────────────────────┘
```

### What is a Block?
- The smallest unit of data in Spark
- Types: RDD partition, Shuffle block, Broadcast block, Task result
- Identified by BlockId: `rdd_0_1` (partition 1 of RDD 0), `shuffle_0_0_0`, `broadcast_0`

In [7]:
# Observe Block Manager: block info for cached RDD

rdd = sc.parallelize(range(10000), 4).map(lambda x: (x, x * 2))
rdd.setName('test_blocks')
rdd.cache()
rdd.count()  # trigger caching

# Check StorageInfo
for rdd_info in sc._jsc.sc().getRDDStorageInfo():
    print(f'RDD: {rdd_info.name()}')
    print(f'  Partitions:        {rdd_info.numPartitions()}')
    print(f'  Cached partitions: {rdd_info.numCachedPartitions()}')
    print(f'  Memory used:       {rdd_info.memSize() / 1024:.1f} KB')
    print(f'  Disk used:         {rdd_info.diskSize() / 1024:.1f} KB')

rdd.unpersist()

print('''
💡 Block Manager key points:
   - All data is managed in Block units
   - Memory Store: cache, broadcast (fast access)
   - Disk Store: spill, shuffle files (large data)
   - BlockTransferService: transfers blocks between nodes (shuffle read)
''')

RDD: test_blocks
  Partitions:        4
  Cached partitions: 4
  Memory used:       68.9 KB
  Disk used:         0.0 KB

💡 Block Manager key points:
   - All data is managed in Block units
   - Memory Store: cache, broadcast (fast access)
   - Disk Store: spill, shuffle files (large data)
   - BlockTransferService: transfers blocks between nodes (shuffle read)



In [8]:
# Broadcast variable block propagation flow

print('=== Broadcast Variable Propagation ===')
print('''
1. Create broadcast variable in Driver
   bc = sc.broadcast(data)

2. Store broadcast_0 block in Driver's BlockManager

3. When an Executor first accesses it:
   ┌────────┐     ┌──────────┐     ┌──────────┐
   │ Driver │────→│Executor 1│     │Executor 2│
   │ bc_0   │     │ bc_0 copy│────→│ bc_0 copy│
   └────────┘     └──────────┘     └──────────┘
   
   BitTorrent style: Driver → some Executors → remaining Executors
   (direct download from Driver by all Executors would bottleneck)

4. Subsequent Tasks on the same Executor use the local copy
''')

# Create an actual broadcast variable and check its size
large_dict = {i: f'value_{i}' * 100 for i in range(10000)}
bc = sc.broadcast(large_dict)

# Task that uses the broadcast variable
result = sc.parallelize(range(100), 4) \
    .map(lambda x: bc.value.get(x % 10000, 'missing')) \
    .count()

print(f'Broadcast variable result: {result} rows')
print(f'\nCheck the broadcast block size in Spark UI > Storage tab.')

bc.unpersist()

=== Broadcast Variable Propagation ===

1. Create broadcast variable in Driver
   bc = sc.broadcast(data)

2. Store broadcast_0 block in Driver's BlockManager

3. When an Executor first accesses it:
   ┌────────┐     ┌──────────┐     ┌──────────┐
   │ Driver │────→│Executor 1│     │Executor 2│
   │ bc_0   │     │ bc_0 copy│────→│ bc_0 copy│
   └────────┘     └──────────┘     └──────────┘
   
   BitTorrent style: Driver → some Executors → remaining Executors
   (direct download from Driver by all Executors would bottleneck)

4. Subsequent Tasks on the same Executor use the local copy

Broadcast variable result: 100 rows

Check the broadcast block size in Spark UI > Storage tab.


---
## 6. Shuffle Internal Mechanics

```
Map Stage (Shuffle Write)              Reduce Stage (Shuffle Read)

Executor 1                             Executor 1
┌─────────────┐                        ┌─────────────┐
│ Task 0      │                        │ Task 3      │
│ partition 0 │                        │             │
│   ↓ sort    │                        │  fetch from │
│   ↓ bucket  │  ───── Shuffle ────→   │  all map    │
│ ┌─┬─┬─┐    │   (network transfer)   │  tasks      │
│ │0│1│2│    │                        │             │
│ └─┴─┴─┘    │                        └─────────────┘
│ (per reduce)│
└─────────────┘

Shuffle Write: each Map Task sorts data by reduce partition ID and writes to local disk
Shuffle Read:  each Reduce Task fetches its partition data from all Map Tasks
```

In [9]:
# Observe the Shuffle process
random.seed(42)
shuffle_data = [(i, random.choice(['A','B','C','D','E']), random.randint(1, 1000))
                for i in range(500_000)]
shuffle_df = spark.createDataFrame(shuffle_data, ['id', 'group', 'value'])

# Query that triggers Shuffle
result = shuffle_df.groupBy('group') \
    .agg(
        F.count('*').alias('cnt'),
        F.sum('value').alias('total'),
        F.avg('value').alias('avg')
    ) \
    .orderBy('group')

result.show()

print('''
🔍 Checking Shuffle in Spark UI:

Jobs tab → target Job → Stages

  Stage 0 (Map):
    - Shuffle Write: buckets data by reduce key
    - Check Shuffle Write size
    
  Stage 1 (Reduce):
    - Shuffle Read: fetches Map stage data over network
    - Check Shuffle Read size / duration

Stage details → Summary Metrics:
    - Shuffle Read Blocked Time: time waiting on network
    - Shuffle Remote Bytes Read: amount fetched from remote nodes
    - Shuffle Local Bytes Read: amount read from local disk
''')

+-----+------+--------+------------------+
|group|   cnt|   total|               avg|
+-----+------+--------+------------------+
|    A|100244|50199999| 500.7780914568453|
|    B| 99236|49583094| 499.6482526502479|
|    C|100555|50383779| 501.0569240714037|
|    D| 99853|49817762|498.91101919822137|
|    E|100112|50167838| 501.1171288157264|
+-----+------+--------+------------------+


🔍 Checking Shuffle in Spark UI:

Jobs tab → target Job → Stages

  Stage 0 (Map):
    - Shuffle Write: buckets data by reduce key
    - Check Shuffle Write size
    
  Stage 1 (Reduce):
    - Shuffle Read: fetches Map stage data over network
    - Check Shuffle Read size / duration

Stage details → Summary Metrics:
    - Shuffle Read Blocked Time: time waiting on network
    - Shuffle Remote Bytes Read: amount fetched from remote nodes
    - Shuffle Local Bytes Read: amount read from local disk



In [10]:
# Inside the Sort-based Shuffle Manager
print('''
=== Spark's Shuffle Manager (SortShuffleManager) ===

Shuffle Write phase:
  1. Buffer Map Task output in memory
  2. When buffer is full, sort by reduce partition ID
  3. Spill sorted data to disk
  4. Produce a single data file + index file
     - Data file: all reduce partitions' data (contiguous)
     - Index file: byte offset for each partition

  File layout:
  shuffle_0_0_0.data   [Part0 data][Part1 data][Part2 data]...
  shuffle_0_0_0.index  [0, 1024, 2048, 3000, ...]  (byte offsets)

Shuffle Read phase:
  1. Reduce Task uses BlockStoreShuffleReader
  2. Looks up its partition offset from each Map Task's index file
  3. Fetches only that range (local or remote)
  4. Merge-sorts in memory if needed

Relevant settings:
  spark.shuffle.file.buffer        = 32k  (write buffer)
  spark.reducer.maxSizeInFlight    = 48m  (concurrent fetch size)
  spark.shuffle.io.maxRetries      = 3    (retries on fetch failure)
  spark.shuffle.compress           = true (snappy compression)
''')


=== Spark's Shuffle Manager (SortShuffleManager) ===

Shuffle Write phase:
  1. Buffer Map Task output in memory
  2. When buffer is full, sort by reduce partition ID
  3. Spill sorted data to disk
  4. Produce a single data file + index file
     - Data file: all reduce partitions' data (contiguous)
     - Index file: byte offset for each partition

  File layout:
  shuffle_0_0_0.data   [Part0 data][Part1 data][Part2 data]...
  shuffle_0_0_0.index  [0, 1024, 2048, 3000, ...]  (byte offsets)

Shuffle Read phase:
  1. Reduce Task uses BlockStoreShuffleReader
  2. Looks up its partition offset from each Map Task's index file
  3. Fetches only that range (local or remote)
  4. Merge-sorts in memory if needed

Relevant settings:
  spark.shuffle.file.buffer        = 32k  (write buffer)
  spark.reducer.maxSizeInFlight    = 48m  (concurrent fetch size)
  spark.shuffle.io.maxRetries      = 3    (retries on fetch failure)
  spark.shuffle.compress           = true (snappy compression)



---
## 7. Lineage and Fault Recovery

Spark recovers from failures not by replicating data, but through **Lineage (the computation history)**.

```
textFile → flatMap → map → reduceByKey → collect
  RDD0      RDD1     RDD2     RDD3

If partition 1 of RDD2 is lost:
  1. Trace back the Lineage
  2. Recompute partition 1 of RDD1
  3. Recompute from partition 1 of RDD0
  4. Partition 1 of RDD2 is recovered
  
  → Only the lost partition is recomputed, not the entire dataset!
```

In [11]:
# Lineage depth and checkpointing

# (a) 20 chained .map() — in PySpark these are FUSED into a single PythonRDD, so the
#     lineage does NOT grow per map. (Consecutive narrow Python ops are pipelined.)
rdd = sc.parallelize(range(1000), 4)
for _ in range(20):
    rdd = rdd.map(lambda x: x + 1)
lin_a = rdd.toDebugString().decode('utf-8')
print(f'(a) 20 chained maps -> {lin_a.count(chr(10)) + 1} lineage lines, '
      f'MapPartitionsRDD count = {lin_a.count("MapPartitionsRDD")}')
print('    => consecutive narrow Python transformations are FUSED into one PythonRDD,')
print('       so chaining maps does NOT lengthen the lineage.\n')
print(lin_a)

# (b) Iterative SHUFFLES (think ML / graph iterations) DO grow the lineage — every wide
#     dependency adds a ShuffledRDD and a new stage. This is where lineage gets dangerously long.
rdd2 = sc.parallelize(range(1000), 4).map(lambda x: (x % 10, x))
for _ in range(8):
    rdd2 = rdd2.reduceByKey(lambda a, b: a + b).map(lambda kv: (kv[0], kv[1] + 1))
lin_b = rdd2.toDebugString().decode('utf-8')
print(f'\n(b) 8 iterative reduceByKey -> ShuffledRDD count = {lin_b.count("ShuffledRDD")} '
      f'({lin_b.count(chr(10)) + 1} lineage lines)')
print('    => the lineage grows with every wide dependency; long chains slow recovery')
print('       and risk a stack overflow while building the DAG.')

print('''
⚠️ When lineage becomes too long (iterative ML / graph algorithms):
   - Recovery must recompute from the start → slow
   - Risk of stack overflow during DAG construction

Solution: break the lineage with checkpoint()
   sc.setCheckpointDir('/tmp/checkpoint')
   rdd.checkpoint()   # materializes to disk and CLEARS the lineage

   cache() vs checkpoint():
   ┌────────────┬──────────────────┬──────────────────┐
   │            │ cache()          │ checkpoint()     │
   ├────────────┼──────────────────┼──────────────────┤
   │ Storage    │ memory (+disk)   │ HDFS/disk        │
   │ Lineage    │ preserved        │ cleared          │
   │ Recovery   │ recompute        │ read from disk   │
   │ Use case   │ optimize reuse   │ cut long lineage │
   └────────────┴──────────────────┴──────────────────┘
''')

(a) 20 chained maps -> 2 lineage lines, MapPartitionsRDD count = 0
    => consecutive narrow Python transformations are FUSED into one PythonRDD,
       so chaining maps does NOT lengthen the lineage.

(4) PythonRDD[89] at RDD at PythonRDD.scala:53 []
 |  ParallelCollectionRDD[88] at readRDDFromFile at PythonRDD.scala:289 []

(b) 8 iterative reduceByKey -> ShuffledRDD count = 8 (34 lineage lines)
    => the lineage grows with every wide dependency; long chains slow recovery
       and risk a stack overflow while building the DAG.

⚠️ When lineage becomes too long (iterative ML / graph algorithms):
   - Recovery must recompute from the start → slow
   - Risk of stack overflow during DAG construction

Solution: break the lineage with checkpoint()
   sc.setCheckpointDir('/tmp/checkpoint')
   rdd.checkpoint()   # materializes to disk and CLEARS the lineage

   cache() vs checkpoint():
   ┌────────────┬──────────────────┬──────────────────┐
   │            │ cache()          │ checkpoint() 

In [12]:
# Speculative Execution
print('''
=== Speculative Execution ===

Problem: a straggler (slow Task) delays the entire Stage

Task 0: ████████ done (2s)
Task 1: ████████ done (2s)
Task 2: ████████████████████████████ slow! (15s)
Task 3: ████████ done (2s)

→ Entire Stage = 15s (bottlenecked by Task 2)

Solution: Speculative Execution
  - Detects Tasks significantly slower than the median
  - Launches a duplicate of the same Task on a different node
  - Uses whichever copy finishes first

Settings:
  spark.speculation                = false (default)
  spark.speculation.interval       = 100ms (check interval)
  spark.speculation.multiplier     = 1.5   (trigger when >1.5x median)
  spark.speculation.quantile       = 0.75  (start after 75% of Tasks complete)

⚠️ Cautions:
  - Dangerous for operations with side effects (DB writes, etc.)
  - Uses more resources
  - If Data Skew is the root cause, Salting is a better solution
''')


=== Speculative Execution ===

Problem: a straggler (slow Task) delays the entire Stage

Task 0: ████████ done (2s)
Task 1: ████████ done (2s)
Task 2: ████████████████████████████ slow! (15s)
Task 3: ████████ done (2s)

→ Entire Stage = 15s (bottlenecked by Task 2)

Solution: Speculative Execution
  - Detects Tasks significantly slower than the median
  - Launches a duplicate of the same Task on a different node
  - Uses whichever copy finishes first

Settings:
  spark.speculation                = false (default)
  spark.speculation.interval       = 100ms (check interval)
  spark.speculation.multiplier     = 1.5   (trigger when >1.5x median)
  spark.speculation.quantile       = 0.75  (start after 75% of Tasks complete)

⚠️ Cautions:
  - Dangerous for operations with side effects (DB writes, etc.)
  - Uses more resources
  - If Data Skew is the root cause, Salting is a better solution



---
## 8. Spark UI Reading Guide

In [13]:
# Run a complex query for monitoring
random.seed(42)
n = 500_000
departments = ['Engineering', 'Marketing', 'Sales', 'HR', 'Finance']

emp_data = [(i, f'emp_{i}', random.choice(departments), random.randint(50000, 150000))
            for i in range(n)]
emp_df = spark.createDataFrame(emp_data, ['id', 'name', 'dept', 'salary'])

dept_data = [(d, f'{d} Division', random.choice(['US','EU','APAC']))
             for d in departments]
dept_df = spark.createDataFrame(dept_data, ['dept', 'full_name', 'region'])

# Complex analytical query
analysis = emp_df \
    .join(F.broadcast(dept_df), 'dept') \
    .groupBy('region', 'dept') \
    .agg(
        F.count('*').alias('headcount'),
        F.avg('salary').alias('avg_salary'),
        F.percentile_approx('salary', 0.5).alias('median_salary'),
    ) \
    .orderBy('region', F.col('avg_salary').desc())

analysis.show()

print('''
🔍 Spark UI Reading Guide (http://localhost:4040)

1. Jobs tab
   - Each Action = 1 Job
   - Number of Stages in Job = number of Shuffles + 1
   - Duration: total elapsed time

2. Stages tab
   - Input Size: amount of data read
   - Shuffle Write/Read: amount of Shuffle data
   - Task count: parallelism of the Stage

3. Stage detail → Summary Metrics
   - Duration: min/25th/median/75th/max
     → max >> median signals Data Skew!
   - Shuffle Read: Remote vs Local
   - GC Time: needs tuning if >10%

4. Stage detail → Event Timeline
   - Blue: computation time
   - Green: Shuffle Read time
   - Orange: Scheduler Delay

5. SQL tab
   - DAG visualization
   - Row count and size per node
   - Exchange = Shuffle point

6. Storage tab
   - Cached RDDs/DataFrames
   - Per-partition distribution

7. Executors tab
   - Memory usage (Storage/Non-storage)
   - GC Time
   - Shuffle Read/Write
   - Task success/failure count
''')

+------+-----------+---------+------------------+-------------+
|region|       dept|headcount|        avg_salary|median_salary|
+------+-----------+---------+------------------+-------------+
|  APAC|  Marketing|    99115|100039.22857286989|        99973|
|  APAC|    Finance|   100018| 100018.1421244176|       100021|
|  APAC|         HR|    99802|  99894.1331235847|        99742|
|    US|      Sales|   100639| 100027.9949621916|       100117|
|    US|Engineering|   100426|100003.76593710792|       100073|
+------+-----------+---------+------------------+-------------+


🔍 Spark UI Reading Guide (http://localhost:4040)

1. Jobs tab
   - Each Action = 1 Job
   - Number of Stages in Job = number of Shuffles + 1
   - Duration: total elapsed time

2. Stages tab
   - Input Size: amount of data read
   - Shuffle Write/Read: amount of Shuffle data
   - Task count: parallelism of the Stage

3. Stage detail → Summary Metrics
   - Duration: min/25th/median/75th/max
     → max >> median signals D

---
## 📝 Key Takeaways

| Concept | Description |
|---------|-------------|
| **Job** | One created per Action |
| **Stage** | Split at Shuffle (Wide Dependency) boundaries |
| **Task** | Execution unit that processes one partition |
| **DAG Scheduler** | Converts Lineage → Stage DAG, analyzes dependencies |
| **Task Scheduler** | Assigns Tasks based on data locality |
| **Narrow Dependency** | 1:1 mapping, pipelining possible, same Stage |
| **Wide Dependency** | N:N Shuffle, Stage boundary |
| **BlockManager** | Block-level data management (memory/disk) |
| **Lineage** | Recovers from failures via recomputation |
| **Checkpoint** | Breaks Lineage and persists to disk |
| **Speculation** | Counters stragglers: duplicate Task on another node |

### Debugging Thought Flow
```
Job is slow
  → Which Stage is slow? (Stages tab)
    → Too many Shuffles? (Shuffle Read/Write size)
    → High variance in Task duration? (Summary Metrics: max >> median)
      → Yes: Data Skew → Salting / AQE
      → No: uniformly slow → adjust partition count / memory / cores
    → High GC Time? (Executors tab)
      → Too much cache / insufficient memory → change StorageLevel / add memory
```

### Next Step (Step 10)
- Tungsten engine & Whole-Stage Code Generation
- Binary memory management
- Analyzing generated Java code

In [14]:
spark.stop()
print('SparkSession stopped')

SparkSession stopped
